In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


COAD_TPM = Path("data/coad/tpm.csv")
READ_TPM = Path("data/read/tpm.csv")
READ_ROW_DATA = Path("data/read/row_data.csv")
UCEC_TPM = Path("data/UCEC/tpm_gene_name.csv")

CRC_OUT = Path("data/CRC_TCGA_LOG2TPM.csv")
UCEC_OUT = Path("data/UCEC_TCGA_LOG2TPM.csv")


def short_barcode(barcode):
    """
    TCGA-AG-3608-01A-01R-... -> TCGA-AG-3608-01A

    Also removes pandas duplicate suffixes such as:
    TCGA-BK-A0CC-01A.1 -> TCGA-BK-A0CC-01A
    """
    barcode = str(barcode)
    barcode = re.sub(r"\.\d+$", "", barcode)

    fields = barcode.split("-")
    if barcode.startswith("TCGA-") and len(fields) >= 4:
        return "-".join(fields[:4])

    return barcode


def load_tpm(path):
    """
    Load a gene-by-sample TPM matrix, retain primary tumors,
    shorten barcodes, and keep the first duplicated sample.
    """
    tpm = pd.read_csv(path, index_col=0)
    tpm = tpm.apply(pd.to_numeric, errors="coerce").fillna(0)

    tpm.columns = [short_barcode(x) for x in tpm.columns]

    
    keep = [
        x.startswith("TCGA-")
        and len(x.split("-")) >= 4
        and x.split("-")[3][:2] == "01"
        for x in tpm.columns
    ]
    tpm = tpm.loc[:, keep]

    duplicated = tpm.columns.duplicated(keep="first")

    if duplicated.any():
        print(
            f"{path.name}: removing {duplicated.sum()} duplicated samples:"
        )
        print(tpm.columns[duplicated].tolist())

    return tpm.loc[:, ~duplicated]


def collapse_gene_symbols(tpm):
    """
    Collapse duplicated gene-symbol rows by summing TPM values.
    """
    tpm = tpm.copy()
    tpm.index = tpm.index.astype(str).str.strip()

    duplicated = tpm.index.duplicated(keep=False)
    if duplicated.any():
        print(
            f"Collapsing {duplicated.sum()} rows belonging to "
            "duplicated gene symbols."
        )

    return tpm.groupby(level=0, sort=False).sum()


def load_read_mapping(path):
    """
    Create Ensembl ID -> HUGO symbol mapping from row_data.csv.
    Only protein-coding genes are retained.
    """
    annotation = pd.read_csv(path, index_col=0)

    required = {"gene_id", "gene_type", "gene_name"}
    missing = required.difference(annotation.columns)
    if missing:
        raise KeyError(
            f"Missing row_data columns: {sorted(missing)}"
        )

    annotation = annotation[
        annotation["gene_type"] == "protein_coding"
    ].copy()

    annotation["ensembl"] = (
        annotation["gene_id"]
        .astype(str)
        .str.replace(r"\.\d+$", "", regex=True)
    )

    annotation["symbol"] = annotation["gene_name"].astype(str)

    annotation = annotation[
        annotation["symbol"].notna()
        & annotation["symbol"].ne("")
        & annotation["symbol"].ne("nan")
    ]

    
    annotation = annotation.drop_duplicates(
        subset="ensembl",
        keep="first",
    )

    return annotation.set_index("ensembl")["symbol"]


def map_read_to_symbols(read_tpm, ensembl_to_symbol):
    """
    Convert READ Ensembl IDs to HUGO symbols and sum duplicated symbols.
    """
    read_tpm = read_tpm.copy()

    ensembl_ids = (
        read_tpm.index.astype(str)
        .str.replace(r"\.\d+$", "", regex=True)
    )

    
    
    symbols = pd.Series(
        ensembl_ids.map(ensembl_to_symbol),
        index=read_tpm.index,
        dtype="object",
    )

    keep = symbols.notna() & (symbols != "")

    print(
        f"READ: mapped {keep.sum():,} of "
        f"{len(read_tpm):,} Ensembl rows."
    )

    read_tpm = read_tpm.loc[keep].copy()
    read_tpm.index = symbols.loc[keep].to_numpy()

    return collapse_gene_symbols(read_tpm)


def save_for_celligner(tpm, output_path):
    """
    Convert gene-by-sample TPM to sample-by-gene log2(TPM + 1).
    """
    result = np.log2(tpm + 1).T

    if result.index.duplicated().any():
        raise ValueError("Duplicated sample IDs remain.")

    if result.columns.duplicated().any():
        raise ValueError("Duplicated gene symbols remain.")

    result.to_csv(output_path)

    print(
        f"Saved {output_path}\n"
        f"  samples: {result.shape[0]:,}\n"
        f"  genes:   {result.shape[1]:,}"
    )






ensembl_to_symbol = load_read_mapping(READ_ROW_DATA)
protein_coding_symbols = set(ensembl_to_symbol.values)






coad = load_tpm(COAD_TPM)
coad = collapse_gene_symbols(coad)


coad = coad.loc[coad.index.isin(protein_coding_symbols)]

read = load_tpm(READ_TPM)
read = map_read_to_symbols(read, ensembl_to_symbol)

crc = pd.concat(
    [coad, read],
    axis=1,
    join="outer",
).fillna(0)


crc = crc.loc[:, ~crc.columns.duplicated(keep="first")]


crc = crc.loc[crc.sum(axis=1) > 0]

save_for_celligner(crc, CRC_OUT)






ucec = load_tpm(UCEC_TPM)
ucec = collapse_gene_symbols(ucec)


ucec = ucec.loc[ucec.index.isin(protein_coding_symbols)]


ucec = ucec.loc[ucec.sum(axis=1) > 0]

save_for_celligner(ucec, UCEC_OUT)